# Classification Track (Part A) — Notebook 10
# 1. Project Introduction  ·  2. Load Dataset

**Course:** 23CSE301 Machine Learning — Capstone Project (Review 1)
**Track:** Classification Part A — Adult / Census Income
**Output:** `artifacts/clf_10_raw.joblib`

---

> ### This is a completely separate machine-learning problem.
>
> The Regression track (Notebooks 01–09) and this Classification track share
> **nothing**: not the data, not the split, not the preprocessing, not the
> metrics. The two datasets are never merged or joined. Everything below
> starts again from the raw files.

---
## 1. Project Introduction

### 1.1 Problem statement

The US Census Bureau collects detailed demographic and employment information
from households. Analysts and policymakers frequently want to understand which
combinations of education, occupation, working hours and household structure
are associated with higher earnings — and to identify, from a survey record
alone, whether a person is likely to fall above or below a given income
threshold.

The question here is: **given a person's demographic and employment
attributes, does their annual income exceed $50,000?**

### 1.2 Objective

1. Audit and clean the Adult dataset, including the missing values it hides
   behind `?` placeholders.
2. Build a leakage-free preprocessing pipeline with a **stratified** split.
3. Train the **five** required Part-A algorithms on one identical split.
4. Report **Accuracy, weighted F1 and a confusion matrix** for each.
5. Analyse class imbalance, errors, and what the false positives and false
   negatives actually mean for a person.

### 1.3 Target variable

**`income`** — a binary label with two values:

* `<=50K` — encoded as **0** (the negative / majority class)
* `>50K` — encoded as **1** (the positive / minority class)

The $50,000 threshold refers to 1994 US dollars, the year the survey extract
was taken.

### 1.4 Why classification is appropriate

The target is a **label from a fixed set of two categories**, not a quantity.
There is no meaningful sense in which one prediction can be "closer" to the
truth than another — a prediction is either right or wrong. That rules out
regression metrics entirely: R², RMSE and MAE all assume a numeric distance
between prediction and truth. The appropriate measures are ones built on
counts of correct and incorrect labels: accuracy, precision, recall, F1 and
the confusion matrix.

### 1.5 Dataset source

* **Name:** Adult Data Set (also known as "Census Income")
* **Repository:** UCI Machine Learning Repository
* **URL:** <https://archive.ics.uci.edu/dataset/2/adult>
* **Donated by:** Ronny Kohavi and Barry Becker (1996)
* **Extracted from:** 1994 US Census database
* **Files used:** `dataset/adult.data` (32,561 rows) and `dataset/adult.test` (16,281 rows)

### 1.6 Real-world relevance

| Application | How an income classifier is used |
|---|---|
| Public policy | Identify demographic groups where earnings lag, to target interventions |
| Census methodology | Impute or validate income responses that are missing or implausible |
| Social research | Quantify how education, occupation and hours relate to earnings |
| ML education | A standard benchmark for imbalanced binary classification |

### 1.7 An ethical note that belongs in the analysis, not a footnote

This dataset contains `race`, `sex`, `native-country`, `marital-status` and
`relationship`. A model trained on it will learn the earnings disparities
present in 1994 American society, including those produced by discrimination.

This has two consequences we take seriously throughout:

1. **Predictive accuracy is not fairness.** A model can be highly accurate
   precisely *because* it has learned a historical injustice. Accuracy alone
   cannot detect this.
2. **This model must not be used for consequential decisions about
   individuals** — lending, hiring, insurance pricing. Doing so would recycle
   1994's inequities into present-day outcomes.

We use the dataset as the assignment specifies, as an academic benchmark, and
return to the issue with measured evidence in Notebook 14.

### 1.8 Dataset description

| # | Column | Type | Meaning |
|---|---|---|---|
| 1 | `age` | numeric | Age in years |
| 2 | `workclass` | categorical | Employment sector (Private, Self-emp, Federal-gov, …) |
| 3 | `fnlwgt` | numeric | "Final weight" — census sampling weight |
| 4 | `education` | categorical | Highest education level (16 values) |
| 5 | `education-num` | numeric | The same education level as an ordinal number |
| 6 | `marital-status` | categorical | Marital status (7 values) |
| 7 | `occupation` | categorical | Occupation category (14 values) |
| 8 | `relationship` | categorical | Role in household (Husband, Wife, Own-child, …) |
| 9 | `race` | categorical | Race (5 values) |
| 10 | `sex` | categorical | Female / Male |
| 11 | `capital-gain` | numeric | Income from investment gains |
| 12 | `capital-loss` | numeric | Losses from investments |
| 13 | `hours-per-week` | numeric | Usual hours worked per week |
| 14 | `native-country` | categorical | Country of origin (41 values) |
| 15 | `income` | **TARGET** | `<=50K` or `>50K` |

---
## 2. Load Dataset

In [1]:
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import sklearn


def find_src_folder() -> Path:
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "src" / "ml_utils.py").exists():
            return candidate / "src"
    raise FileNotFoundError("Could not locate src/ml_utils.py — check the repo layout.")


sys.path.insert(0, str(find_src_folder()))

from ml_utils import (  # noqa: E402
    DATA_DIR,
    RANDOM_STATE,
    ROOT,
    audit_table,
    save_artifact,
    section,
    set_plot_style,
)

warnings.filterwarnings("ignore", category=FutureWarning)
set_plot_style()

print(f"Project root : {ROOT}")
print(f"pandas {pd.__version__} | numpy {np.__version__} | "
      f"scikit-learn {sklearn.__version__} | seaborn {sns.__version__}")

Project root : /home/claude/project
pandas 3.0.2 | numpy 2.4.4 | scikit-learn 1.8.0 | seaborn 0.13.2


### 2.1 Reading the raw files

The Adult files need three non-obvious arguments, and getting any of them
wrong corrupts the data silently:

1. **`header=None`** — the files contain no header row. Without this, the
   first respondent's record would be consumed as column names.
2. **`skipinitialspace=True`** — values are separated by `", "`, not `","`.
   Without this, every category would carry a leading space and
   `" Private"` would be a different string from `"Private"`.
3. **`na_values="?"`** — missing values are stored as the literal `?`. Without
   this, pandas would report zero missing values and treat `?` as a legitimate
   employment category. Notebook 02's placeholder check in the regression
   track is what tells us to look for exactly this.

`adult.test` has two further quirks: a junk first line, and a trailing full
stop on every label (`<=50K.` rather than `<=50K`).

In [2]:
COLUMN_NAMES = [
    "age", "workclass", "fnlwgt", "education", "education-num",
    "marital-status", "occupation", "relationship", "race", "sex",
    "capital-gain", "capital-loss", "hours-per-week", "native-country", "income",
]

READ_OPTIONS = dict(
    header=None,
    names=COLUMN_NAMES,
    skipinitialspace=True,   # strip the space after each comma
    na_values="?",           # '?' is this dataset's missing-value marker
)

train_file = pd.read_csv(DATA_DIR / "adult.data", **READ_OPTIONS)
test_file = pd.read_csv(DATA_DIR / "adult.test", skiprows=1, **READ_OPTIONS)

section("THE TWO SOURCE FILES")
print(f"adult.data : {train_file.shape[0]:,} rows")
print(f"adult.test : {test_file.shape[0]:,} rows")
print()
print("Label values as they appear in each file:")
print(f"  adult.data : {sorted(train_file['income'].unique())}")
print(f"  adult.test : {sorted(test_file['income'].unique())}   <-- note the trailing '.'")

THE TWO SOURCE FILES
adult.data : 32,561 rows
adult.test : 16,281 rows

Label values as they appear in each file:
  adult.data : ['<=50K', '>50K']
  adult.test : ['<=50K.', '>50K.']   <-- note the trailing '.'


### 2.2 Fixing the label formatting and combining the files

In [3]:
# Strip the trailing full stop that appears only in adult.test.
test_file["income"] = test_file["income"].str.rstrip(".")

adult_df = pd.concat([train_file, test_file], ignore_index=True)

section("COMBINED DATASET")
print(f"Total rows : {adult_df.shape[0]:,}")
print(f"Columns    : {adult_df.shape[1]}")
print(f"Label values after cleaning: {sorted(adult_df['income'].unique())}")

COMBINED DATASET
Total rows : 48,842
Columns    : 15
Label values after cleaning: ['<=50K', '>50K']


### Why combine the two files rather than use them as the split?

UCI ships a pre-made train/test division. We combine them and create our own
split in Notebook 13 instead, for three reasons:

1. **The assignment requires a stratified split with `random_state=42`**, so
   that the split is reproducible and the class balance is preserved in both
   halves. The supplied division guarantees neither.
2. **One split must be shared by all five algorithms.** Making it ourselves
   from one combined pool is the cleanest way to guarantee that.
3. More data is available for both training and evaluation — 48,842 rows
   rather than 32,561.

The two files are two random samples from the same 1994 census extract, not
different populations, so combining them introduces no distribution shift. We
verify that below rather than assuming it.

In [4]:
section("ARE THE TWO FILES COMPARABLE?")
comparison = pd.DataFrame({
    "adult.data": train_file["income"].value_counts(normalize=True).mul(100).round(2),
    "adult.test": test_file["income"].value_counts(normalize=True).mul(100).round(2),
})
print("Class proportions (%):")
print(comparison.to_string())
print()
print("Mean of the numeric columns in each file:")
numeric_cols = train_file.select_dtypes(include=np.number).columns
print(pd.DataFrame({
    "adult.data": train_file[numeric_cols].mean(),
    "adult.test": test_file[numeric_cols].mean(),
}).round(2).to_string())

ARE THE TWO FILES COMPARABLE?
Class proportions (%):
        adult.data  adult.test
income                        
<=50K        75.92       76.38
>50K         24.08       23.62

Mean of the numeric columns in each file:
                adult.data  adult.test
age                  38.58       38.77
fnlwgt           189778.37   189435.68
education-num        10.08       10.07
capital-gain       1077.65     1081.91
capital-loss         87.30       87.90
hours-per-week       40.44       40.39


**Observation.** The class proportions agree to within a fraction of a
percentage point, and the means of every numeric column are close.

**Interpretation.** The two files are samples from the same population. There
is no distribution shift, so combining them is legitimate and the pooled
dataset is coherent.

### 2.3 `head()`

In [5]:
adult_df.head()

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


### 2.4 `tail()`

In [6]:
adult_df.tail()

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
48837,39,Private,215419,Bachelors,13,Divorced,Prof-specialty,Not-in-family,White,Female,0,0,36,United-States,<=50K
48838,64,NaN,321403,HS-grad,9,Widowed,NaN,Other-relative,Black,Male,0,0,40,United-States,<=50K
48839,38,Private,374983,Bachelors,13,Married-civ-spouse,Prof-specialty,Husband,White,Male,0,0,50,United-States,<=50K
48840,44,Private,83891,Bachelors,13,Divorced,Adm-clerical,Own-child,Asian-Pac-Islander,Male,5455,0,40,United-States,<=50K
48841,35,Self-emp-inc,182148,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,60,United-States,>50K


### 2.5 Shape, columns and data types

In [7]:
section("SHAPE")
print(f"Rows    : {adult_df.shape[0]:,}")
print(f"Columns : {adult_df.shape[1]}")

section("COLUMN NAMES")
for i, col in enumerate(adult_df.columns, start=1):
    print(f"{i:>2}. {col}")

section("DATA TYPES")
print(adult_df.dtypes.to_string())

SHAPE
Rows    : 48,842
Columns : 15
COLUMN NAMES
 1. age
 2. workclass
 3. fnlwgt
 4. education
 5. education-num
 6. marital-status
 7. occupation
 8. relationship
 9. race
10. sex
11. capital-gain
12. capital-loss
13. hours-per-week
14. native-country
15. income
DATA TYPES
age               int64
workclass           str
fnlwgt            int64
education           str
education-num     int64
marital-status      str
occupation          str
relationship        str
race                str
sex                 str
capital-gain      int64
capital-loss      int64
hours-per-week    int64
native-country      str
income              str


### 2.6 `info()`

In [8]:
adult_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 48842 entries, 0 to 48841
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   age             48842 non-null  int64
 1   workclass       46043 non-null  str  
 2   fnlwgt          48842 non-null  int64
 3   education       48842 non-null  str  
 4   education-num   48842 non-null  int64
 5   marital-status  48842 non-null  str  
 6   occupation      46033 non-null  str  
 7   relationship    48842 non-null  str  
 8   race            48842 non-null  str  
 9   sex             48842 non-null  str  
 10  capital-gain    48842 non-null  int64
 11  capital-loss    48842 non-null  int64
 12  hours-per-week  48842 non-null  int64
 13  native-country  47985 non-null  str  
 14  income          48842 non-null  str  
dtypes: int64(6), str(9)
memory usage: 5.6 MB


**Observation.** Three columns have fewer non-null entries than the 48,842
total: `workclass`, `occupation` and `native-country`. Six columns are numeric;
nine are text.

**Interpretation.** Those three gaps only appear because we passed
`na_values="?"` when reading. Without that argument, pandas would have reported
a perfectly complete dataset and we would have trained models on `?` as though
it were a real occupation. Notebook 11 investigates the pattern of the
missingness before deciding how to handle it.

### 2.7 `describe()` — numeric columns

In [9]:
adult_df.describe().T.round(2)

,count,mean,std,min,25%,50%,75%,max
age,48842.0,38.64,13.71,17.0,28.0,37.0,48.0,90.0
fnlwgt,48842.0,189664.13,105604.03,12285.0,117550.5,178144.5,237642.0,1490400.0
education-num,48842.0,10.08,2.57,1.0,9.0,10.0,12.0,16.0
capital-gain,48842.0,1079.07,7452.02,0.0,0.0,0.0,0.0,99999.0
capital-loss,48842.0,87.50,403.00,0.0,0.0,0.0,0.0,4356.0
hours-per-week,48842.0,40.42,12.39,1.0,40.0,40.0,45.0,99.0


**Observation.**

* `age` spans 17 to 90 years, which is plausible for a survey of working-age
  adults and above.
* `capital-gain` has a median of 0 but a maximum of 99,999 — the top value
  looks like a coded ceiling rather than a real amount.
* `capital-loss` is similarly zero for most respondents.
* `hours-per-week` ranges from 1 to 99 with a mean near 40.
* `fnlwgt` is in the hundreds of thousands and has nothing to do with the
  individual's income.

**Interpretation.** `capital-gain` and `capital-loss` are **severely
zero-inflated**, the same pattern rainfall showed in the regression track.
The `99999` value is almost certainly a top-coded placeholder — Notebook 11
checks how many records carry it. `fnlwgt` needs a decision of its own, taken
in Notebook 13.

### 2.8 `describe()` — categorical columns

In [10]:
adult_df.describe(include="object").T

/tmp/ipykernel_313/1802143076.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  adult_df.describe(include="object").T


,count,unique,top,freq
workclass,46043,8,Private,33906
education,48842,16,HS-grad,15784
marital-status,48842,7,Married-civ-spouse,22379
occupation,46033,14,Prof-specialty,6172
relationship,48842,6,Husband,19716
race,48842,5,White,41762
sex,48842,2,Male,32650
native-country,47985,41,United-States,43832
income,48842,2,<=50K,37155


### 2.9 Quick per-column audit

In [11]:
audit_table(adult_df)

,Column,Dtype,Missing,Missing %,Unique,Example value
0,age,int64,0,0.000,74,39
1,workclass,str,2799,5.731,8,State-gov
2,fnlwgt,int64,0,0.000,28523,77516
3,education,str,0,0.000,16,Bachelors
4,education-num,int64,0,0.000,16,13
5,marital-status,str,0,0.000,7,Never-married
6,occupation,str,2809,5.751,14,Adm-clerical
7,relationship,str,0,0.000,6,Not-in-family
8,race,str,0,0.000,5,White
9,sex,str,0,0.000,2,Male


---
## Handover to Notebook 11

The combined DataFrame is saved with **no rows removed and no values changed**
beyond the two formatting repairs that were required to read the files
correctly (stripping the trailing full stop from the test labels, and reading
`?` as missing).

In [12]:
save_artifact(adult_df, "clf_10_raw")

print(f"\nShape passed forward: {adult_df.shape}")
print("\nNotebook 10 complete.")
print("Next: 11_classification_data_quality_audit.ipynb")

[saved]  artifacts/clf_10_raw.joblib

Shape passed forward: (48842, 15)

Notebook 10 complete.
Next: 11_classification_data_quality_audit.ipynb
